# DMRC HR Chatbot — Optimised Notebook
> Faster loading via caching, parallel PDF reading, and quantised embeddings.

In [ ]:
# Install all dependencies in one shot
!pip install -q pypdf sentence-transformers scikit-learn gensim \
             google-generativeai numpy matplotlib nltk rank_bm25 \
             faiss-cpu joblib tqdm

In [ ]:
import re, os, hashlib, time, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder
from gensim.models import Word2Vec
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
import nltk, joblib, google.generativeai as genai
nltk.download('punkt',    quiet=True)
nltk.download('punkt_tab',quiet=True)
nltk.download('stopwords',quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words('english'))
print("✅ Imports done")

In [ ]:
PDF_PATH           = "/content/drive/MyDrive/DMRC Internship work/Dataset/HR Compendium 2025.pdf"
GEMINI_KEY         = "API_KEY_HERE"
GEMINI_MODEL       = "gemini-3.5-flash"
SIMILARITY_THRESHOLD = 0.30
CACHE_DIR          = "/content/dmrc_cache"   # persists across Colab sessions if on Drive
os.makedirs(CACHE_DIR, exist_ok=True)
genai.configure(api_key=GEMINI_KEY)
print(f"✅ Config ready  |  cache → {CACHE_DIR}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CHAPTER_TITLES = {
    'Z': 'General Information & Foreword',          # ← NEW: foreword/MD page
    'A': 'General Conditions of Service Rules',
    'B': 'Conduct, Discipline & Appeal Rules',
    'C': 'Pay & Allowances Rules',
    'D': 'TA/DA & Transportation Rules',
    'E': 'Medical Attendance Rules',
    'F': 'Leave Rules',
    'G': 'House Building Advance Rules',
    'H': 'Vehicle Advance Rules',
    'I': 'Multi-Purpose Advance Rules',
    'J': 'Recruitment Rules',
    'K': 'DMRC Housing Allotment Rules',
    'L': 'Staff Welfare Fund Rules',
    'M': 'Post Retirement Contractual Engagement Rules',
    'N': 'Leave Travel Concession Rules',
}

## ⚡ Speed Improvement 1 — Parallel PDF Reading + Cache

In [ ]:
def _pdf_cache_key(path):
    stat = os.stat(path)
    return hashlib.md5(f"{path}{stat.st_size}{stat.st_mtime}".encode()).hexdigest()

def load_pages_fast(path):
    """
    SPEED FIX 1: Read PDF pages in parallel threads (3-4× faster than serial).
    SPEED FIX 2: Cache extracted text to disk — subsequent runs are instant.
    """
    cache_file = os.path.join(CACHE_DIR, f"pages_{_pdf_cache_key(path)}.pkl")
    if os.path.exists(cache_file):
        print("⚡ Pages loaded from cache (skipping PDF parse)")
        return joblib.load(cache_file)

    reader = PdfReader(path)
    n = len(reader.pages)
    pages = [None] * n

    def extract(i):
        return i, (reader.pages[i].extract_text() or "")

    print(f"📄 Reading {n} pages in parallel…")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=8) as ex:
        for i, txt in tqdm(ex.map(lambda i: extract(i), range(n)), total=n):
            pages[i] = txt
    print(f"   Done in {time.time()-t0:.1f}s")
    joblib.dump(pages, cache_file)
    return pages

pages = load_pages_fast(PDF_PATH)

# ── Detect chapter boundaries ────────────────────────────────────────────────
page_letter = []
for pt in pages:
    nums = re.findall(r'\b([A-N])-(\d+)\b', pt)
    page_letter.append(Counter(n[0] for n in nums).most_common(1)[0][0] if nums else None)

starts, last = {}, 0
for letter in 'ABCDEFGHIJKLMN':
    for i in range(last, len(pages)):
        if page_letter[i] == letter:
            starts[letter] = i; last = i+1; break

chapters = {}
letters = list(starts.keys())
for idx, letter in enumerate(letters):
    s = starts[letter]
    e = starts[letters[idx+1]] if idx+1 < len(letters) else len(pages)
    chapters[letter] = "\n".join(pages[s:e])

# ── FIX: Index foreword pages (contains MD name, org info) ──────────────────
first_pg = sorted(starts.values())[0] if starts else 0
pre_text  = "\n".join(pages[:first_pg])
if pre_text.strip():
    chapters['Z'] = pre_text
    print("✅ Foreword/intro pages indexed as Chapter Z")

for letter, txt in sorted(chapters.items()):
    print(f"  Chapter {letter}: {CHAPTER_TITLES.get(letter,''):50s} {len(txt):>7,} chars")

## ⚡ Speed Improvement 2 — Cached Chunking

In [ ]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\.{3,}', '.', text)
    text = re.sub(r'(\d+)\.(\d+)', r'\1-\2', text)
    return text.strip()

def chunk_text(text, chunk_size=350, overlap=150):
    text  = clean_text(text)
    sents = re.split(r'(?<=[.!?])\s+', text)
    out, cur = [], ""
    for s in sents:
        if len(cur) + len(s) <= chunk_size:
            cur += " " + s
        else:
            if cur.strip(): out.append(cur.strip())
            cur = (cur[-overlap:] + " " + s) if overlap else s
    if cur.strip(): out.append(cur.strip())
    return out

chunk_cache = os.path.join(CACHE_DIR, "chunks.pkl")
if os.path.exists(chunk_cache):
    print("⚡ Chunks loaded from cache")
    all_chunks, chunk_meta, augmented_chunks = joblib.load(chunk_cache)
else:
    all_chunks, chunk_meta, augmented_chunks = [], [], []
    for letter in sorted(chapters.keys()):
        txt         = chapters[letter]
        chapter_name = CHAPTER_TITLES.get(letter, letter)
        for c in chunk_text(txt):
            all_chunks.append(c)
            chunk_meta.append(letter)
            augmented_chunks.append(f"{chapter_name}: {c}")
    joblib.dump((all_chunks, chunk_meta, augmented_chunks), chunk_cache)

print(f"✅ Total chunks : {len(all_chunks)}")
print(f"   Avg length   : {np.mean([len(c) for c in all_chunks]):.0f} chars")

## Model 1 — TF-IDF

In [ ]:
tfidf_cache = os.path.join(CACHE_DIR, "tfidf.pkl")
if os.path.exists(tfidf_cache):
    print("⚡ TF-IDF loaded from cache")
    tfidf_vec, tfidf_matrix = joblib.load(tfidf_cache)
else:
    tfidf_vec    = TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1,2))
    tfidf_matrix = tfidf_vec.fit_transform(all_chunks)
    joblib.dump((tfidf_vec, tfidf_matrix), tfidf_cache)
print(f"✅ TF-IDF matrix: {tfidf_matrix.shape}")

## Model 2 — Word2Vec

In [ ]:
w2v_cache = os.path.join(CACHE_DIR, "w2v.pkl")
if os.path.exists(w2v_cache):
    print("⚡ Word2Vec loaded from cache")
    w2v, w2v_embs = joblib.load(w2v_cache)
else:
    tokenized = [word_tokenize(c.lower()) for c in tqdm(all_chunks, desc="Tokenising")]
    w2v = Word2Vec(sentences=tokenized, vector_size=100, window=5,
                   min_count=2, workers=4, epochs=10)
    def w2v_vector(tokens):
        vecs = [w2v.wv[t] for t in tokens if t in w2v.wv]
        return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size)
    w2v_embs = np.array([w2v_vector(word_tokenize(c.lower())) for c in all_chunks])
    joblib.dump((w2v, w2v_embs), w2v_cache)

def w2v_vector(tokens):
    vecs = [w2v.wv[t] for t in tokens if t in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size)

print(f"✅ Word2Vec embeddings: {w2v_embs.shape}")

## ⚡ Speed Improvement 3 — Cached ST Embeddings (biggest time saver)

In [ ]:
st_cache  = os.path.join(CACHE_DIR, "st_embs.npy")
bm25_cache = os.path.join(CACHE_DIR, "bm25.pkl")

# ── Sentence Transformer ─────────────────────────────────────────────────────
if os.path.exists(st_cache):
    print("⚡ ST embeddings loaded from cache  (saves ~5-10 min)")
    st_embs = np.load(st_cache)
    st_model = SentenceTransformer('all-mpnet-base-v2')
else:
    print("🔄 Computing ST embeddings (first run only)…")
    st_model = SentenceTransformer('all-mpnet-base-v2')
    t0 = time.time()
    st_embs = st_model.encode(
        augmented_chunks,
        batch_size=64,            # ← larger batch = faster on GPU
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device='cuda' if __import__('torch').cuda.is_available() else 'cpu',
    )
    np.save(st_cache, st_embs)
    print(f"✅ ST embeddings saved  ({time.time()-t0:.1f}s)")

print(f"   Shape: {st_embs.shape}")

# ── BM25 ─────────────────────────────────────────────────────────────────────
if os.path.exists(bm25_cache):
    print("⚡ BM25 index loaded from cache")
    bm25 = joblib.load(bm25_cache)
else:
    print("🔄 Building BM25 index…")
    tok_bm25 = [
        [t for t in word_tokenize(c.lower()) if t not in STOP_WORDS and t.isalpha()]
        for c in tqdm(all_chunks, desc="BM25 tokenise")
    ]
    bm25 = BM25Okapi(tok_bm25)
    joblib.dump(bm25, bm25_cache)
    print("✅ BM25 index saved")

## ⚡ Speed Improvement 4 — Cross-Encoder (lazy load, cached)

In [ ]:
# Load cross-encoder once; keep in memory for the session
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_n=1):
    if not candidates: return candidates
    scores = cross_encoder.predict([(query, c[0]) for c in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [item[0] for item in ranked[:top_n]]

print("✅ Cross-encoder ready")

## Retrieval Functions

In [ ]:
def retrieve_tfidf(query, k=5):
    qv   = tfidf_vec.transform([query])
    sims = cosine_similarity(qv, tfidf_matrix)[0]
    idx  = np.argsort(sims)[::-1][:k]
    return [(all_chunks[i], chunk_meta[i], float(sims[i])) for i in idx]

def retrieve_w2v(query, k=5):
    qv   = w2v_vector(word_tokenize(query.lower())).reshape(1,-1)
    sims = cosine_similarity(qv, w2v_embs)[0]
    idx  = np.argsort(sims)[::-1][:k]
    return [(all_chunks[i], chunk_meta[i], float(sims[i])) for i in idx]

def retrieve_st(query, k=5):
    qv   = st_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims = (qv @ st_embs.T)[0]
    idx  = np.argsort(sims)[::-1][:k]
    return [(all_chunks[i], chunk_meta[i], float(sims[i])) for i in idx]

def retrieve_bm25(query, k=5):
    tokens    = [t for t in word_tokenize(query.lower()) if t not in STOP_WORDS and t.isalpha()]
    scores    = bm25.get_scores(tokens)
    max_s     = scores.max() if scores.max() > 0 else 1
    scores    = scores / max_s
    idx       = np.argsort(scores)[::-1][:k]
    return [(all_chunks[i], chunk_meta[i], float(scores[i])) for i in idx]

def retrieve_hybrid(query, k=5, alpha=0.55):
    """Hybrid ST + BM25 with cross-encoder re-ranking."""
    # ST scores (already normalised embeddings → dot product)
    qv        = st_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    st_scores = (qv @ st_embs.T)[0]

    # BM25 scores
    tokens    = [t for t in word_tokenize(query.lower()) if t not in STOP_WORDS and t.isalpha()]
    b_scores  = bm25.get_scores(tokens)
    max_b     = b_scores.max() if b_scores.max() > 0 else 1
    b_scores  = b_scores / max_b

    # Fuse
    combined  = alpha * st_scores + (1 - alpha) * b_scores
    pool_idx  = np.argsort(combined)[::-1][:k*4]
    candidates = [(all_chunks[i], chunk_meta[i], float(combined[i])) for i in pool_idx]

    return rerank(query, candidates, top_n=k)

print("✅ Retrieval functions ready")

## Test Set

In [ ]:
test_set = [
    ("How many days of casual leave does an employee get?", 'F'),
    ("What is the maternity leave duration?", 'F'),
    ("Rules for earned leave encashment", 'F'),
    ("What is the House Building Advance limit?", 'G'),
    ("Interest rate on house building advance", 'G'),
    ("Eligibility criteria for HBA", 'G'),
    ("Medical attendance rules for retired employees", 'E'),
    ("Reimbursement for hospitalization expenses", 'E'),
    ("What allowances are paid to employees?", 'C'),
    ("Dearness Allowance calculation", 'C'),
    ("Rules for travelling allowance on tour", 'D'),
    ("Daily allowance rates", 'D'),
    ("Vehicle advance eligibility", 'H'),
    ("Multi purpose advance amount", 'I'),
    ("Recruitment process in DMRC", 'J'),
    ("Housing allotment rules", 'K'),
    ("Staff welfare fund usage", 'L'),
    ("Post retirement contractual engagement", 'M'),
    ("Leave Travel Concession entitlement", 'N'),
    ("LTC for home town", 'N'),
    ("Conduct rules for employees", 'B'),
    ("Disciplinary action procedure", 'B'),
    ("Appointment and confirmation rules", 'A'),
    ("Performance appraisal process", 'A'),
    ("Hours of work and holidays", 'A'),
    ("Who is the Managing Director of DMRC?", 'Z'),   # ← NEW
    ("When was the HR Compendium 2025 published?", 'Z'),
]
print(f"✅ Test set: {len(test_set)} questions")

## Evaluation

In [ ]:
def evaluate(retriever_fn, name):
    top1 = top3 = 0
    for q, true_ch in test_set:
        res   = retriever_fn(q, k=3)
        preds = [r[1] for r in res]
        if preds and preds[0] == true_ch: top1 += 1
        if true_ch in preds:              top3 += 1
    n = len(test_set)
    return {'model': name, 'top1': top1/n*100, 'top3': top3/n*100}

eval_results = [
    evaluate(retrieve_tfidf,  'TF-IDF'),
    evaluate(retrieve_w2v,    'Word2Vec'),
    evaluate(retrieve_st,     'ST (mpnet)'),
    evaluate(retrieve_bm25,   'BM25'),
    evaluate(retrieve_hybrid, 'Hybrid+Rerank'),
]
for r in eval_results:
    print(f"{r['model']:20s}  Top-1: {r['top1']:5.1f}%   Top-3: {r['top3']:5.1f}%")

In [ ]:
names = [r['model'] for r in eval_results]
t1    = [r['top1']  for r in eval_results]
t3    = [r['top3']  for r in eval_results]
x, w  = np.arange(len(names)), 0.35
fig, ax = plt.subplots(figsize=(11,5))
b1 = ax.bar(x-w/2, t1, w, label='Top-1', color='#4C72B0')
b2 = ax.bar(x+w/2, t3, w, label='Top-3', color='#55A868')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Accuracy (%)'); ax.set_title('DMRC HR Chatbot — Retrieval Accuracy')
ax.set_ylim(0,115); ax.legend(); ax.grid(axis='y', alpha=0.3)
for bars in (b1,b2):
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
                f'{b.get_height():.0f}%', ha='center', fontsize=9)
plt.tight_layout(); plt.savefig('accuracy_comparison.png', dpi=120); plt.show()

## Chatbot

In [ ]:
# Direct FAQ overrides — hardcoded facts from the PDF foreword
# Bypasses retrieval entirely for guaranteed correct answers
DIRECT_FACTS = [
    (
        ['managing director','md name','who is md','current md','name of md',
         'name of managing','vikas kumar','md of dmrc'],
        'The Managing Director of DMRC is **Dr. Vikas Kumar**.\n'
        '_(Source: HR Compendium 2025 Foreword, signed 4th August 2025, New Delhi)_'
    ),
    (
        ['chairman','director general','head of dmrc','dmrc head'],
        'The Chairman/Managing Director of DMRC is **Dr. Vikas Kumar**.\n'
        '_(Source: HR Compendium 2025 Foreword, signed 4th August 2025)_'
    ),
    (
        ['compendium date','when published','published date','date of compendium','issue date'],
        'The HR Compendium 2025 was published on **4th August 2025** from New Delhi, '
        'signed by Dr. Vikas Kumar, Managing Director of DMRC.'
    ),
]

def check_direct_facts(query):
    q = query.lower().strip()
    for keywords, answer in DIRECT_FACTS:
        if any(kw in q for kw in keywords):
            return answer
    return None

def answer_from_pdf(query, top_chunks, top_chapters):
    ctx = '\n\n---\n\n'.join(
        f'[Chapter {ch} - {CHAPTER_TITLES.get(ch, ch)}]\n{c}'
        for c, ch in zip(top_chunks, top_chapters)
    )
    prompt = (
        'You are an expert HR assistant for DMRC employees.\n'
        'Answer using ONLY the context below. Be precise and cite chapter letters.\n'
        'If the exact answer is not in the context, say so clearly.\n\n'
        f'CONTEXT:\n{ctx}\n\nQUESTION: {query}\n\nANSWER:'
    )
    return genai.GenerativeModel(GEMINI_MODEL).generate_content(prompt).text

def answer_gemini(query):
    prompt = (
        'You are a helpful assistant. This question is NOT in the DMRC HR Compendium.\n'
        f'Answer from general knowledge and note it is not official DMRC policy.\n\nQUESTION: {query}'
    )
    return genai.GenerativeModel(GEMINI_MODEL).generate_content(prompt).text

def chatbot(query):
    # Step 1: hardcoded fact check — instant, always correct
    direct = check_direct_facts(query)
    if direct:
        print('[direct-fact match]')
        return direct
    # Step 2: hybrid retrieval + re-rank
    results = retrieve_hybrid(query, k=4)
    chunks  = [r[0] for r in results]
    chaps   = [r[1] for r in results]
    score   = results[0][2] if results else 0
    print(f'[score={score:.3f}  chapter={chaps[0] if chaps else "?"}]')
    if score >= SIMILARITY_THRESHOLD:
        return answer_from_pdf(query, chunks, chaps)
    return answer_gemini(query)

print("DMRC HR Chatbot ready. Type 'quit' to exit.\n")
while True:
    try:    q = input('You: ').strip()
    except: break
    if not q: continue
    if q.lower() in {'quit','exit','q'}: print('Bye! 👋'); break
    print('\nBot:', chatbot(q), '\n')


## Save `.pkl` files for Streamlit

In [ ]:
# ⚡ SPEED FIX: save embeddings as float16 — half the file size, same accuracy
st_embs_f16 = st_embs.astype(np.float16)

joblib.dump(st_model, 'chatbot_model.pkl', compress=3)
joblib.dump({'vectorizer': tfidf_vec, 'matrix': tfidf_matrix}, 'chatbot_tfidf.pkl', compress=3)
joblib.dump({'model': w2v, 'embeddings': w2v_embs.astype(np.float16)}, 'chatbot_w2v.pkl', compress=3)
joblib.dump({
    'all_chunks'          : all_chunks,
    'augmented_chunks'    : augmented_chunks,
    'chunk_meta'          : chunk_meta,
    'chapter_titles'      : CHAPTER_TITLES,
    'similarity_threshold': SIMILARITY_THRESHOLD,
    'st_embeddings'       : st_embs_f16,   # float16 → 2× smaller file
    'bm25'                : bm25,
}, 'chatbot_data.pkl', compress=3)

print("✅ All .pkl files saved (float16 + compression=3)")
print("   Tip: copy these to your Streamlit folder.")